# core

> Lightweight extensions to python-docx: markdown rendering, markdown-to-runs parsing, and tracked changes (insertions/deletions with author attribution).

In [ ]:
#| default_exp core

In [ ]:
#| export
from copy import deepcopy
from datetime import datetime, timezone
from docx import Document as DocxDocument
from docx.document import Document
from docx.text.run import Run
from docx.text.paragraph import Paragraph
from docx.table import Table, _Cell, _Row
from docx.oxml import OxmlElement
from docx.oxml.table import CT_Tbl
from docx.oxml.ns import qn
from docx.shared import Emu, Inches, Pt
from fastcore.basics import *
from mistletoe import Document as MdDocument
from mistletoe.span_token import SpanToken, RawText, Strong, Emphasis
from mistletoe import span_token

import re

In [ ]:
#| export
_docx_tracking = None

def set_tracking(author=None):
    "Set author name to enable tracked changes, or None to disable"
    global _docx_tracking
    _docx_tracking = author

class Revision:
    "Tracked change metadata: typ ('ins'/'del'), author, date"
    def __init__(self, typ, author, date): store_attr()
    def __repr__(self): return f'Revision(typ={self.typ!r}, author={self.author!r}, date={self.date!r})'

def _revision(el):
    if el is None: return None
    return Revision(el.tag.split('}')[-1], el.get(qn('w:author')), el.get(qn('w:date')))

@patch(as_prop=True)
def revision(self:Run):
    "Tracked change metadata from parent w:ins/w:del, or None"
    p = self._r.getparent()
    tag = p.tag.split('}')[-1] if p is not None else None
    return _revision(p) if tag in ('ins','del') else None

@patch(as_prop=True)
def text(self:Run):
    "Get run text from w:t or w:delText"
    return ''.join(el.text for el in self._r
                   if el.tag.split('}')[-1] in ('t', 'delText') and el.text)

@patch(set_prop=True)
def text(self:Run, value): self._r.text = value

@patch
def _repr_markdown_(self:Run):
    "Render run as markdown with formatting and revision spans"
    t = self.text
    if not t: return ''
    b, i, u = self.font.bold, self.font.italic, self.font.underline
    rev = getattr(self, 'revision', None)
    if not b and not i and not u and not rev: return t
    lead, trail = len(t)-len(t.lstrip()), len(t)-len(t.rstrip())
    inner = t.strip()
    if not inner: return t
    pre, suf = t[:lead], (t[len(t)-trail:] if trail else '')
    if   b and i: inner = f'***{inner}***'
    elif b:       inner = f'**{inner}**'
    elif i:       inner = f'*{inner}*'
    if u: inner = f'<u>{inner}</u>'
    res = f'{pre}{inner}{suf}'
    if rev:
        cls = 'deletion' if rev.typ == 'del' else 'insertion'
        res = f'<span class="{cls}">{res}</span>'
    return res

In [ ]:
doc = DocxDocument()
p = doc.add_paragraph()
run = p.add_run('Hello world')
run

<div class="prose">

Hello world

</div>

In [ ]:
run.font.bold = True
run

<div class="prose">

**Hello world**

</div>

In [ ]:
#| export
@patch(as_prop=True)
def runs(self:Paragraph):
    "All runs in document order, including those inside w:ins and w:del"
    runs = []
    for el in self._p:
        tag = el.tag.split('}')[-1]
        if tag == 'r': runs.append(Run(el, self))
        elif tag in ('ins', 'del'):
            for r_el in el.findall(qn('w:r')): runs.append(Run(r_el, self))
    return runs

@patch(as_prop=True)
def revision(self:Paragraph):
    "Tracked change metadata from paragraph mark w:ins/w:del, or None"
    pPr = self._p.find(qn('w:pPr'))
    if pPr is None: return None
    rPr = pPr.find(qn('w:rPr'))
    if rPr is None: return None
    for tag in ('ins','del'):
        rev = rPr.find(qn(f'w:{tag}'))
        if rev is not None: return _revision(rev)

def _rev_el(typ, author):
    "Create a w:ins or w:del element with author and timestamp"
    el = OxmlElement(f'w:{typ}')
    el.set(qn('w:id'), str(id(el)))
    el.set(qn('w:author'), author)
    el.set(qn('w:date'), datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'))
    return el

@patch
def add_run(self:Paragraph, text=None, style=None):
    "Add a run, wrapping in w:ins if tracking is set"
    run = self._orig_add_run(text, style)
    if _docx_tracking:
        el = _rev_el('ins', _docx_tracking)
        run._r.addprevious(el)
        el.append(run._r)
    return run

@patch
def _repr_markdown_(self:Paragraph):
    "Render paragraph as markdown with style tag and optional indent"
    list_style = ''
    if self.style.name == 'List Number': list_style = '1. '
    elif self.style.name == 'List Bullet': list_style = '- '
    runs = self.runs
    if not any(r.text.strip() for r in runs): return ''
    return list_style+''.join(r._repr_markdown_() for r in runs)

In [ ]:
set_tracking('Author Name')
run = p.add_run('. A new run!')
run.italic = True
p

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

</div>

In [ ]:
run.revision

Revision(typ='ins', author='Author Name', date='2026-05-07T21:02:29Z')

In [ ]:
#| export
class Underline(SpanToken):
    pattern = re.compile(r'<u>(.+?)</u>')
    parse_inner = True
    parse_group = 1

span_token.add_token(Underline)

def _walk(node, para, bold=False, italic=False, underline=False):
    if isinstance(node, RawText):
        run = para.add_run(node.content)
        if bold: run.font.bold = True
        if italic: run.font.italic = True
        if underline: run.font.underline = True
        return
    for child in node.children: _walk(child, para, bold or isinstance(node, Strong), italic or isinstance(node, Emphasis), underline or isinstance(node, Underline))

@patch
def add_md(self:Paragraph, text):
    doc = MdDocument(text)
    for block in doc.children:
        for child in block.children: _walk(child, self)
    return self

def _mark_para_inserted(p):
    if not _docx_tracking: return p
    pPr = p._p.get_or_add_pPr()
    rPr = pPr.find(qn('w:rPr')) or OxmlElement('w:rPr')
    if rPr.getparent() is None: pPr.append(rPr)
    if rPr.find(qn('w:ins')) is None: rPr.append(_rev_el('ins', _docx_tracking))
    return p

@patch
def add_paragraph(self:Document, text='', style=None):
    p = _mark_para_inserted(self._orig_add_paragraph(style=style))
    if text: p.add_md(text)
    return p

In [ ]:
set_tracking(None)
new_p = doc.add_paragraph('*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>')
new_p

<div class="prose">

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>

</div>

In [ ]:
#| export
@patch
def _repr_markdown_(self:Document):
    "Render full document as markdown"
    res = []
    for el in self.element.body:
        tag = el.tag.split('}')[-1]
        if   tag == 'p':   md = Paragraph(el, self)._repr_markdown_()
        elif tag == 'tbl': md = Table(el, self)._repr_markdown_()
        elif tag in ('ins', 'del'):
            cls = 'insertion' if tag == 'ins' else 'deletion'
            for child in el:
                ctag = child.tag.split('}')[-1]
                if   ctag == 'p':   inner = Paragraph(child, self)._repr_markdown_()
                elif ctag == 'tbl': inner = Table(child, self)._repr_markdown_()
                else: continue
                if inner: res.append(f'<span class="{cls}">{inner}</span>')
            continue
        else: continue
        if md: res.append(md)
    return '\n\n'.join(res)

In [ ]:
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>

</div>

In [ ]:
#| export
def _get_rPr_font(para):
    "Get font rPr element from paragraph's existing runs, or pPr/rPr fallback"
    for r in para._p.findall(qn('w:r')):
        rPr = r.find(qn('w:rPr'))
        if rPr is not None: return rPr
    pPr = para._p.find(qn('w:pPr'))
    if pPr is not None: return pPr.find(qn('w:rPr'))

_font_tags = {qn(t) for t in (
    'w:rStyle', 'w:rFonts', 'w:sz', 'w:szCs', 'w:color'
)}

def _base_rPr_from_para(para):
    "Base character formatting from paragraph, excluding emphasis"
    src = _get_rPr_font(para)
    if src is None: return None
    rPr = OxmlElement('w:rPr')
    for child in src:
        if child.tag in _font_tags: rPr.append(deepcopy(child))
    return rPr

def _apply_base_rPr(run, base_rPr):
    if base_rPr is None: return
    run_rPr = run._r.get_or_add_rPr()
    for child in base_rPr:
        existing = run_rPr.find(child.tag)
        if existing is not None: run_rPr.remove(existing)
        run_rPr.append(deepcopy(child))

@patch
def apply_base(self:Paragraph, src):
    src_pPr = src._p.find(qn('w:pPr'))
    if src_pPr is not None:
        dst_pPr = self._p.get_or_add_pPr()
        for child in src_pPr:
            existing = dst_pPr.find(child.tag)
            if existing is not None: dst_pPr.remove(existing)
            dst_pPr.append(deepcopy(child))
    base_rPr = _base_rPr_from_para(src)
    for r in self.runs: _apply_base_rPr(r, base_rPr)
    return self

In [ ]:
#| export
def _content_width(part):
    "Content width (page minus margins) from the document's last section"
    doc = part.document
    section = doc.sections[-1]
    pw = section.page_width  or Inches(8.5)
    lm = section.left_margin or Inches(1)
    rm = section.right_margin or Inches(1)
    return Emu(pw - lm - rm)

def _mark_table_inserted(tbl):
    if not _docx_tracking: return tbl
    for row in tbl.rows:
        if _row_rev(row, 'ins') is None:
            _get_or_add_trPr(row._tr).append(_rev_el('ins', _docx_tracking))
    return tbl

def _insert_block(anchor_el, parent, content, after, para_base=None):
    pos = anchor_el.addnext if after else anchor_el.addprevious
    if isinstance(content, tuple):
        rows, cols = content
        el = CT_Tbl.new_tbl(rows, cols, _content_width(parent.part))
        pos(el)
        return _mark_table_inserted(Table(el, parent))
    el = anchor_el.makeelement(qn('w:p'), {})
    pos(el)
    p = _mark_para_inserted(Paragraph(el, parent))
    if para_base is not None: p.apply_base(para_base)
    p.add_md(content)
    return p

@patch
def insert_before(self:Paragraph, content): return _insert_block(self._p, self._parent, content, False, self)

@patch
def insert_after(self:Paragraph, content): return _insert_block(self._p, self._parent, content, True, self)

In [ ]:
set_tracking(None)
for i in range(1, 4): doc.add_paragraph(f'Item {i}', style='List Number')
paras = doc.paragraphs
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>

1. Item 1

1. Item 2

1. Item 3

</div>

In [ ]:
set_tracking('Author Name')
paras[-3].insert_after('Inserting after 1!')
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>

1. Item 1

1. <span class="insertion">Inserting after 1!</span>

1. Item 2

1. Item 3

</div>

In [ ]:
#| export
@patch
def add_md(self:_Cell, text):
    p = self.paragraphs[0]
    p.add_md(text)
    return self

def _get_or_add_trPr(tr):
    trPr = tr.find(qn('w:trPr'))
    if trPr is None:
        trPr = OxmlElement('w:trPr')
        tr.insert(0, trPr)
    return trPr

@patch
def add_row(self:Table, *cells):
    cells = listify(cells)
    row = self._orig_add_row()
    if _docx_tracking: _get_or_add_trPr(row._tr).append(_rev_el('ins', _docx_tracking))
    
    n = len(row.cells)
    for i,val in enumerate(cells):
        cell = row.cells[i].merge(row.cells[n-1]) if i==len(cells)-1 and len(cells)<n else row.cells[i]
        cell.add_md(str(val))
    return row

def _row_cells(row): return row._tr.findall(qn('w:tc'))

def _tc_md(tc):
    "Render a table XML cell as inline markdown"
    cell = _Cell(tc, None)
    parts = [r._repr_markdown_() for p in cell.paragraphs for r in p.runs]
    return ' '.join(parts).strip().replace('\n', ' ')

@patch
def _repr_markdown_(self:Table):
    "Render table as markdown without duplicating merged cells"
    rows = [[_tc_md(tc) for tc in _row_cells(row)] for row in self.rows]
    if not rows: return ''
    hdr = '| ' + ' | '.join(rows[0]) + ' |'
    sep = '| ' + ' | '.join('---' for _ in rows[0]) + ' |'
    body = '\n'.join('| ' + ' | '.join(r) + ' |' for r in rows[1:])
    return f'{hdr}\n{sep}\n{body}'

In [ ]:
tbl = doc.add_table(rows=0, cols=3)
tbl.add_row('This is a merged header')
for i in range(3):
    tbl.add_row(f'R{i}C0', f'R{i}C1', f'R{i}C2')
tbl

<div class="prose">

| <span class="insertion">This is a merged header</span> |
| --- |
| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
#| export
@patch
def insert_before(self:Table, content): return _insert_block(self._tbl, self._parent, content, False)

@patch
def insert_after(self:Table, content): return _insert_block(self._tbl, self._parent, content, True)

In [ ]:
set_tracking(None)
tbl2 = tbl.insert_before((0, 2))
tbl2.add_row('Apples', 'Bananas')
tbl2.insert_after('Some random *text*')
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>

1. Item 1

1. <span class="insertion">Inserting after 1!</span>

1. Item 2

1. Item 3

| Apples | Bananas |
| --- | --- |


Some random *text*

| <span class="insertion">This is a merged header</span> |
| --- |
| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
#| export
@patch
def delete(self:Run):
    "Delete run, or mark as tracked deletion if tracking is set"
    parent = self._r.getparent()
    parent_tag = parent.tag.split('}')[-1] if parent is not None else None

    if _docx_tracking:
        if parent_tag == 'del': return
        if parent_tag == 'ins': return parent.getparent().remove(parent)

        el = _rev_el('del', _docx_tracking)
        for t in self._r.iter(qn('w:t')): t.tag = qn('w:delText')
        self._r.addprevious(el)
        el.append(self._r)
    else: self._r.getparent().remove(self._r)

In [ ]:
new_r = new_p.add_run('This is going to be deleted.')
new_p

<div class="prose">

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>This is going to be deleted.

</div>

In [ ]:
set_tracking('Author Name')
new_r.delete()
new_p

<div class="prose">

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u><span class="deletion">This is going to be deleted.</span>

</div>

In [ ]:
new_r.revision

Revision(typ='del', author='Author Name', date='2026-05-07T21:02:30Z')

In [ ]:
#| export
@patch
def delete(self:Paragraph):
    "Delete paragraph, or mark all runs and the paragraph mark as tracked deletion"
    if not _docx_tracking: return self._p.getparent().remove(self._p)

    for run in list(self.runs): run.delete()

    pPr = self._p.get_or_add_pPr()
    rPr = pPr.find(qn('w:rPr'))
    if rPr is None:
        rPr = OxmlElement('w:rPr')
        pPr.append(rPr)

    if rPr.find(qn('w:del')) is None: rPr.append(_rev_el('del', _docx_tracking))

In [ ]:
set_tracking('Author Name')
paras[2].delete()
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u><span class="deletion">This is going to be deleted.</span>

1. <span class="deletion">Item 1</span>

1. <span class="insertion">Inserting after 1!</span>

1. Item 2

1. Item 3

| Apples | Bananas |
| --- | --- |


Some random *text*

| <span class="insertion">This is a merged header</span> |
| --- |
| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
paras[2].revision

Revision(typ='del', author='Author Name', date='2026-05-07T21:02:30Z')

In [ ]:
#| export
def _row_rev(row, typ=None):
    trPr = row._tr.find(qn('w:trPr'))
    if trPr is None: return None
    typs = (typ,) if typ else ('ins','del')
    for t in typs:
        el = trPr.find(qn(f'w:{t}'))
        if el is not None: return el

@patch
def delete(self:_Row):
    "Delete row, or mark row and cell contents as tracked deletions"
    if not _docx_tracking: return self._tr.getparent().remove(self._tr)
    if _row_rev(self, 'del') is not None: return
    if _row_rev(self, 'ins') is not None: return self._tr.getparent().remove(self._tr)
    _get_or_add_trPr(self._tr).append(_rev_el('del', _docx_tracking))
    for cell in self.cells:
        for p in cell.paragraphs: p.delete()

@patch
def delete(self:Table):
    "Delete table, or mark all rows and contents as tracked deletions"
    if not _docx_tracking: return self._tbl.getparent().remove(self._tbl)
    for row in list(self.rows): row.delete()

In [ ]:
set_tracking('Author Name')
tbl2.delete()
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u><span class="deletion">This is going to be deleted.</span>

1. <span class="deletion">Item 1</span>

1. <span class="insertion">Inserting after 1!</span>

1. Item 2

1. Item 3

| <span class="deletion">Apples</span> | <span class="deletion">Bananas</span> |
| --- | --- |


Some random *text*

| <span class="insertion">This is a merged header</span> |
| --- |
| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
#| export
@patch
def _iter_blocks(self:Document):
    for el in self.element.body:
        tag = el.tag.split('}')[-1]
        if tag=='p': yield Paragraph(el, self)
        elif tag=='tbl': yield Table(el, self)

@patch
def __iter__(self:Document): yield from self._iter_blocks()

@patch(as_prop=True)
def blocks(self:Document): return list(self)

In [ ]:
for b in doc: print(b)

<docx.text.paragraph.Paragraph object>
<docx.text.paragraph.Paragraph object>
<docx.text.paragraph.Paragraph object>
<docx.text.paragraph.Paragraph object>
<docx.text.paragraph.Paragraph object>
<docx.text.paragraph.Paragraph object>
<docx.table.Table object>
<docx.text.paragraph.Paragraph object>
<docx.table.Table object>


In [ ]:
#| export
@patch(as_prop=True)
def text(self:Paragraph): return ''.join(r.text for r in self.runs)

@patch(as_prop=True)
def text(self:Table): return '\n'.join('\t'.join(' '.join(p.text for p in c.paragraphs) for c in r.cells) for r in self.rows)

@patch
def search(self:Document, text, regex=False, case=False):
    def match(s):
        if regex: return re.search(text, s, 0 if case else re.I)
        return text in s if case else text.lower() in s.lower()
    return [b for b in self if match(b.text)]

In [ ]:
doc.search('apple')[0]

<div class="prose">

| <span class="deletion">Apples</span> | <span class="deletion">Bananas</span> |
| --- | --- |


</div>

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()